# 8.1D Mini Project: Sydney Housing Price Prediction and Decision Support System

## Setup

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

## Part 1: Data Loading and Cleaning

In [2]:
df = pd.read_csv("sydney_housing.csv", parse_dates=["sold_date"])
df["property_type"] = df["property_type"].replace({"apartment": "unit"})
df["is_strata"] = df["property_type"].isin(["unit", "townhouse"]).astype(int)
df["group"] = np.where(df["is_strata"] == 1, "Strata", "House")
order = ["Cabramatta", "Bankstown", "Marrickville"]

print(df.shape)
print(df.isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'sydney_housing.csv'

## Part 2: Data Understanding and Feature Engineering

#### 2.1 Price distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df["sold_price"] / 1e6, bins=20); ax[0].set_title("Sale price ($M)")
ax[1].hist(np.log(df["sold_price"]), bins=20); ax[1].set_title("Log sale price")
plt.show()

print(df["sold_price"].describe())

**Price distribution**

Sale prices range from 375,000 to 3.5M, with a median of 912,500 and a mean of 1.16M. The distribution is strongly right-skewed, because most sales are units under 1M while a few large houses sell above 2.5M. Taking the log of price gives a much more balanced shape, so the models are trained on log(price).

#### 2.2 Differences between suburbs

In [ ]:
sns.boxplot(data=df, x="suburb", y="sold_price", hue="group", order=order)
plt.title("Sale price by suburb")
plt.show()

df.groupby(["suburb", "group"])["sold_price"].agg(["count", "median"])

**Differences between suburbs**

Property type separates prices more than suburb does. In every suburb, houses sell for roughly two to four times the price of strata properties. Among strata properties, the expected order holds: Cabramatta has the lowest median (502,500), then Bankstown (620,000), then Marrickville (982,500). The house medians do not follow this order. Cabramatta houses (1.95M) sold for more than Bankstown houses (1.46M), because the Cabramatta sample includes several new builds and very large blocks (up to 1,225 m²). This shows the sample does not fully represent each market.

#### 2.3 Trends over time

In [ ]:
sns.scatterplot(data=df, x="sold_date", y="sold_price", hue="suburb")
plt.title("Sale price over time")
plt.xticks(rotation=45)
plt.show()

**Trend over time**

No clear price trend appears. Only Cabramatta covers April to September, while Bankstown and Marrickville sales fall mostly in August and September, so any time effect cannot be separated from the suburb effect.

#### 2.4 Outliers

In [ ]:
q1 = df.groupby(["suburb", "group"])["sold_price"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby(["suburb", "group"])["sold_price"].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
outliers = df[(df["sold_price"] < q1 - 1.5 * iqr) | (df["sold_price"] > q3 + 1.5 * iqr)]
outliers[["address", "suburb", "property_type", "bedrooms", "land_size_m2", "sold_price"]]

**Outliers**

Using the IQR rule within each suburb and property group, seven properties were flagged. All are three-bedroom townhouses or units (780,000 to 985,000) that sit above the typical one- and two-bedroom units in the strata group. They are genuine sales, not errors, so they were kept. The model handles them through the bedrooms feature.

**Expected key variables**

Before feature engineering, I expected these three variables to matter most:

**Property type (house vs strata):** A house includes land, which is the main source of value in Sydney.  
**Suburb:** It captures location, distance to the CBD and transport access.  
**Bedrooms:** More bedrooms usually means a larger home that suits families, and it is the main size measure available for every property, including units where land size is missing or not comparable.

#### 2.5 Feature engineering

In [ ]:
# Car spaces: blank in Marrickville = no parking (0); other blanks = typical value (1)
mar = df["suburb"] == "Marrickville"
df.loc[mar, "car_spaces"] = df.loc[mar, "car_spaces"].fillna(0)
df["car_spaces"] = df["car_spaces"].fillna(1)

# Land only counts for houses; strata = 0; missing house land = median house land
house_land = df.loc[df["is_strata"] == 0, "land_size_m2"].median()
df["land"] = np.where(df["is_strata"] == 1, 0, df["land_size_m2"].fillna(house_land))

X = pd.get_dummies(df[["bedrooms", "bathrooms", "car_spaces", "land", "is_strata", "suburb"]],
                   columns=["suburb"], drop_first=True, dtype=int)
y = np.log(df["sold_price"])   # log price because prices are right-skewed

X.assign(log_price=y).corr()["log_price"].drop("log_price").sort_values(ascending=False).round(2)

## Part 3: Model Development and Evaluation

#### 3.1 Three models with 5-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=42),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
price = df["sold_price"]

results = []
for name, m in models.items():
    cv_pred = np.exp(cross_val_predict(m, X, y, cv=cv))   # predictions on unseen folds
    train_pred = np.exp(m.fit(X, y).predict(X))            # predictions on training data
    df["pred_" + name] = cv_pred
    results.append([name,
                    mean_absolute_error(price, train_pred), mean_absolute_error(price, cv_pred),
                    np.sqrt(mean_squared_error(price, cv_pred)),
                    r2_score(price, train_pred), r2_score(price, cv_pred)])

res = pd.DataFrame(results, columns=["Model", "Train MAE", "CV MAE", "CV RMSE", "Train R2", "CV R2"])
res.round({"Train MAE": 0, "CV MAE": 0, "CV RMSE": 0, "Train R2": 3, "CV R2": 3})

#### 3.2 Model complexity

In [ ]:
rows = []
for d in [1, 2, 3, 5, 8, None]:
    m = RandomForestRegressor(n_estimators=200, max_depth=d, random_state=42)
    cv_pred = np.exp(cross_val_predict(m, X, y, cv=cv))
    train_pred = np.exp(m.fit(X, y).predict(X))
    rows.append([d, mean_absolute_error(price, train_pred), mean_absolute_error(price, cv_pred)])

pd.DataFrame(rows, columns=["max_depth", "Train MAE", "CV MAE"]).round(0)

## Part 4: Investigating Prediction Failures

In [ ]:
best = res.loc[res["CV MAE"].idxmin(), "Model"]
print("Best model:", best)

df["pred"] = df["pred_" + best].round(0)
df["error"] = df["pred"] - df["sold_price"]
df["pct_error"] = (df["error"] / df["sold_price"] * 100).round(1)

cols = ["address", "suburb", "property_type", "bedrooms", "bathrooms", "car_spaces",
        "land_size_m2", "sold_price", "pred", "error", "pct_error"]
df.loc[df["error"].abs().sort_values(ascending=False).index[:5], cols]

## Part 5: Deployment
Saves the best model to `model.joblib` for the Streamlit app (`app.py`).

In [ ]:
import joblib
joblib.dump({"model": models[best].fit(X, y), "columns": list(X.columns), "house_land": house_land},
            "model.joblib")
print("Saved model.joblib using", best)